# Day 5 — Delta Lake: Architecture, Formats & Schema

Plan for about 2–2.5 hours in the notebook, plus any discussion. Work through the parts in order, or split early ingest sections and later practice sections across the class if needed.

Topics include Parquet vs Delta and the transaction log, CSV and JSON loads to Delta with `overwrite` / `append` / `mergeSchema`, partitioned writes, `DESCRIBE DETAIL` and `DESCRIBE HISTORY`, strict vs evolving schema, `replaceWhere`, SQL reads on `delta.` paths, data quality checks, skill-builder cells, and optional challenges.

Prerequisites: `%run ./02-Mount-Azure-Data-Lake`, and the usual course files `2010-summary.csv` and `json/2015-summary.json` on the storage account. In UC-oriented workspaces, prefer `_metadata.file_path` (or a literal path) over `input_file_name()` for file lineage.

Public Databricks training labs on Delta reads/writes, versioning, schema evolution, and optimizations are broadly aligned with this notebook; paths here use ABFS as in the rest of this course.

## Part A — Connect to ADLS (same as Days 1–4)

In [0]:
%run ./02-Mount-Azure-Data-Lake

# Azure Data Lake Storage Gen2 — ABFS + OAuth (no DBFS mount)

This notebook configures **Spark** to read/write Azure Data Lake Storage Gen2 using the **ABFS** URI scheme (`abfss://...`) and **OAuth** via `spark.conf` (Service Principal). **We do not use `dbutils.fs.mount`** here.

---

## Prerequisites

1. ADLS Gen2 storage account and container with your data.
2. Service Principal with **Storage Blob Data Contributor** (or equivalent) on the container.
3. Fill in **tenant_id**, **client_id**, **client_secret** below (or use `dbutils.secrets.get` for production).

**Expected data layout** under the container (folder `data/`):
- `data/json/2015-summary.json`
- `data/2010-summary.csv`

After this runs, `base_path` points at `abfss://<container>@<account>.dfs.core.windows.net/data`.

---

## How lesson notebooks connect

**All hands-on notebooks that read ADLS** (`01-Day1-...`, `01-Day2-...`, `01-Day3-...`, `03-Day3-...`, `01-Day4-...`, `03-Day4-...`, `01-Day5-...`, `03-Day5-...`, `04-Day5-...`) must start with **`%run ./02-Mount-Azure-Data-Lake`** (typically the **first code cell**) so **`spark.conf` OAuth** and **`base_path`** are set on the cluster.

- Keep **`02-Mount-Azure-Data-Lake.ipynb`** in the **same workspace folder** as the lesson notebook, or adjust the path (e.g. `%run ../notebooks/02-Mount-Azure-Data-Lake`).
- After a **new cluster** or **RESTART**, run **`%run`** again, then the lesson **paths** cell (`BASE_PATH = base_path`, etc.).
- You can **Run all** on this notebook alone for debugging; in class it is normally executed **only** via `%run` from other notebooks.

## Step 1 — Storage account and OAuth (Service Principal)

OAuth configured for account: sadbtrng19032026


## Step 2 — Base path (ABFSS) and optional smoke read

base_path: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data


+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
BASE_PATH = base_path
STUDENT_ID = "u01"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
P_JSON = DAY5 + "/delta/flight_json"
P_PART = DAY5 + "/delta/flight_partitioned"
P_STRICT = DAY5 + "/delta/schema_strict"
P_EVOLVE = DAY5 + "/delta/schema_evolve_demo"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"

# Verify source files exist
print("=== Checking Prerequisites ===")
prereqs_met = True
try:
    df_csv = spark.read.format("csv").option("header", True).option("inferSchema", True).load(SOURCE_CSV).limit(1)
    print(f"✓ Source CSV exists: {SOURCE_CSV}")
except Exception as e:
    print(f"✗ Source CSV NOT found: {SOURCE_CSV}")
    print(f"  Error: {type(e).__name__}")
    prereqs_met = False

try:
    df_json = spark.read.format("json").load(SOURCE_JSON).limit(1)
    print(f"✓ Source JSON exists: {SOURCE_JSON}")
except Exception as e:
    print(f"✗ Source JSON NOT found: {SOURCE_JSON}")
    print(f"  Error: {type(e).__name__}")
    prereqs_met = False

if not prereqs_met:
    print("\n⚠️  WARNING: Some prerequisite data files are missing.")
    print("   Delta read/write operations will fail.")

print("\n=== Path Configuration ===")
for _n, _p in [
    ("DAY5 root", DAY5),
    ("basic", P_BASIC),
    ("json delta", P_JSON),
    ("partitioned", P_PART),
    ("strict", P_STRICT),
    ("evolve", P_EVOLVE),
]:
    print(_n + ":", _p)

DAY5 root: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5
basic: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_summary_basic
json delta: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_json
partitioned: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_partitioned
strict: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/schema_strict
evolve: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/schema_evolve_demo


## Part B — Theory: Parquet-only lakes vs Delta

**Plain Parquet directories**
- No single **transaction log** → readers may see **partial writes** if a job fails mid-write.
- **No built-in ACID** semantics across concurrent writers.
- **Schema drift** is painful: files disagree silently.

**Delta Lake**
- **`_delta_log`** records commits (JSON + checkpoints).
- **ACID** table semantics at the directory level.
- **Time travel**, **CLONE**, **MERGE/UPDATE/DELETE**, **Change Data Feed** (later topics / Day 6).
- **Schema enforcement** and controlled **evolution**.

### B2 — Checklist (Delta features you will touch today)

| Feature | Where in Day 5 |
|---------|----------------|
| ACID overwrite / append | Parts C–E |
| Transaction log (history) | Part F |
| Schema enforcement | Part G |
| Schema evolution (`mergeSchema`) | Part G |
| Partitioning | Part E |
| Optimize (Databricks) | Notebook **03** |

### B3 — Inside `_delta_log` (mental model)

Each commit appends **JSON** files (and periodic **checkpoints**) under **`_delta_log/`** at the table root. Readers always see a **consistent snapshot** from the latest successful commit.

| Artifact | Role |
|----------|------|
| `*.json` commit | Add/remove files, schema, metadata, metrics |
| Checkpoints | Speed up log replay for large tables |
| Reader protocol | Uses log to decide which Parquet files are **active** |

**Takeaway:** Delta is **Parquet files + transaction log**, not “magic storage” — governance still lives in **ABFS ACLs** + **Unity Catalog** when you register tables.

## Part C — First Delta table: CSV → Delta (`overwrite`)

In [0]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit

df0 = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
)
df0.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_BASIC)
print("Rows:", df0.count(), "| Path:", P_BASIC)

Rows: 255 | Path: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_summary_basic


In [0]:
spark.read.format("delta").load(P_BASIC).printSchema()
spark.read.format("delta").load(P_BASIC).show(5, truncate=False)

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_path: string (nullable = true)

+-----------------+-------------------+-----+--------------------------+-----------------------------------------------------------------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|ingested_at               |source_path                                                                  |
+-----------------+-------------------+-----+--------------------------+-----------------------------------------------------------------------------+
|United States    |Romania            |1    |2026-03-21 13:53:19.218299|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|
|United States    |Ireland            |264  |2026-03-21 13:53:19.218299|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|
|United Stat

## Part D — Append with `append` + extra column (`mergeSchema`)

In [0]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit, expr

append_batch = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
    .withColumn("batch_id", lit("batch_002"))
    .withColumn("load_comment", lit("append_with_mergeSchema"))
)
append_batch.write.format("delta").mode("append").option("mergeSchema", "true").save(P_BASIC)
print("After append, total rows:", spark.read.format("delta").load(P_BASIC).count())
spark.read.format("delta").load(P_BASIC).printSchema()

After append, total rows: 510
root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_path: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- load_comment: string (nullable = true)



## Part E — JSON → Delta + **partitioned** write (lab *08* style)

In [0]:
from pyspark.sql.functions import col, substring, upper as u

jdf = spark.read.format("json").option("inferSchema", True).load(SOURCE_JSON)
jdf = jdf.withColumn("dest_initial", u(substring(col("DEST_COUNTRY_NAME"), 1, 1)))
jdf.write.format("delta").mode("overwrite").partitionBy("dest_initial").option("overwriteSchema", "true").save(P_PART)
print("Partitioned Delta at:", P_PART)

Partitioned Delta at: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_partitioned


In [0]:
spark.read.format("delta").load(P_PART).groupBy("dest_initial").count().orderBy("dest_initial").show(30)

+------------+-----+
|dest_initial|count|
+------------+-----+
|           A|    9|
|           B|   11|
|           C|   14|
|           D|    4|
|           E|    4|
|           F|    6|
|           G|    9|
|           H|    4|
|           I|    7|
|           J|    3|
|           K|    3|
|           L|    3|
|           M|    7|
|           N|    7|
|           P|    9|
|           Q|    1|
|           R|    2|
|           S|   15|
|           T|    7|
|           U|  129|
|           V|    1|
|           Z|    1|
+------------+-----+



In [0]:
# Read only one partition (predicate pushdown)
spark.read.format("delta").load(P_PART).where("dest_initial = 'U'").select("DEST_COUNTRY_NAME", "count").show(10, truncate=False)

+-----------------+-----+
|DEST_COUNTRY_NAME|count|
+-----------------+-----+
|United States    |15   |
|United States    |1    |
|United States    |344  |
|United States    |62   |
|United States    |1    |
|United States    |62   |
|United States    |325  |
|United States    |39   |
|United States    |6    |
|United States    |1    |
+-----------------+-----+
only showing top 10 rows


## Part F — `DESCRIBE DETAIL` + `DESCRIBE HISTORY`

In [0]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").show(truncate=False)

+------+------------------------------------+----+-----------+--------------------------------------------------------------------------------------------+-----------------------+-------------------+----------------+-----------------+--------+-----------+-------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name|description|location                                                                                    |createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                           |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                                     |clusterByAuto|
+------+------------------------------------+----+-----------+------------------------------------------

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").select("version", "timestamp", "operation", "operationMetrics").show(20, truncate=False)

+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                         |
+-------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------+
|1      |2026-03-21 13:53:36|WRITE    |{numFiles -> 1, numOutputRows -> 255, numOutputBytes -> 6817}                                                                            |
|0      |2026-03-21 13:53:22|WRITE    |{numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 255, numOutputBytes -> 6245}|
+-------+-------------------+---------+-----------------------------------------------------------------------

## Part F2 — SQL on Delta path (Spark SQL `delta.\`path\``)

Same physical table as **`spark.read.format('delta').load(P_BASIC)`** — analysts often use **`delta.\`abfss://...\``** in SQL warehouses.

### F2a — Row count

In [0]:
spark.sql(f"SELECT COUNT(*) AS c FROM delta.`{P_BASIC}`").show(truncate=False)

+---+
|c  |
+---+
|510|
+---+



### F2b — Top destinations

In [0]:
spark.sql(f"SELECT DEST_COUNTRY_NAME, SUM(count) AS s FROM delta.`{P_BASIC}` GROUP BY 1 ORDER BY 2 DESC LIMIT 8").show(truncate=False)

+------------------+------+
|DEST_COUNTRY_NAME |s     |
+------------------+------+
|United States     |769864|
|Canada            |16542 |
|Mexico            |12400 |
|United Kingdom    |3258  |
|Germany           |2784  |
|Japan             |2766  |
|Dominican Republic|2218  |
|Brazil            |1990  |
+------------------+------+



### F2c — Min / max count

In [0]:
spark.sql(f"SELECT MIN(count) AS mn, MAX(count) AS mx, AVG(count) AS av FROM delta.`{P_BASIC}`").show(truncate=False)

+---+------+-----------------+
|mn |mx    |av               |
+---+------+-----------------+
|1  |348113|1655.956862745098|
+---+------+-----------------+



### F2d — Distinct origins

In [0]:
spark.sql(f"SELECT COUNT(DISTINCT ORIGIN_COUNTRY_NAME) AS origins FROM delta.`{P_BASIC}`").show(truncate=False)

+-------+
|origins|
+-------+
|131    |
+-------+



### F2e — Filter + limit

In [0]:
spark.sql(f"SELECT * FROM delta.`{P_BASIC}` WHERE count > 1000 ORDER BY count DESC LIMIT 10").show(truncate=False)

+-----------------+-------------------+------+--------------------------+-----------------------------------------------------------------------------+---------+-----------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count |ingested_at               |source_path                                                                  |batch_id |load_comment           |
+-----------------+-------------------+------+--------------------------+-----------------------------------------------------------------------------+---------+-----------------------+
|United States    |United States      |348113|2026-03-21 13:53:19.218299|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|NULL     |NULL                   |
|United States    |United States      |348113|2026-03-21 13:53:34.479984|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|batch_002|append_with_mergeSchema|
|United States    |Canada             |8305  |2026-03-21 13:53:34.4799

## Part G — Schema **strict** vs **evolution**

In [0]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

strict_schema = StructType(
    [
        StructField("DEST_COUNTRY_NAME", StringType(), True),
        StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
        StructField("count", LongType(), True),
    ]
)
sdf = spark.read.format("csv").option("header", True).schema(strict_schema).load(SOURCE_CSV)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_STRICT)
print("Strict-schema Delta:", P_STRICT)

Strict-schema Delta: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/schema_strict


In [0]:
# Try to append a row with an EXTRA column — expect failure without mergeSchema
from pyspark.sql import Row

bad = spark.createDataFrame([Row(DEST_COUNTRY_NAME="X", ORIGIN_COUNTRY_NAME="Y", count=1, extra_col="oops")])
try:
    bad.write.format("delta").mode("append").save(P_STRICT)
except Exception as e:
    print("Expected failure (schema mismatch):", type(e).__name__)

Expected failure (schema mismatch): AnalysisException


In [0]:
# Same append with mergeSchema = true
bad.write.format("delta").mode("append").option("mergeSchema", "true").save(P_STRICT)
spark.read.format("delta").load(P_STRICT).printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)
 |-- extra_col: string (nullable = true)



## Part H — Multiple **read patterns** (practice)

### H1 — `format('delta').load`

In [0]:
spark.read.format('delta').load(P_BASIC).select('DEST_COUNTRY_NAME', 'count').limit(5).show()

+-----------------+-----+
|DEST_COUNTRY_NAME|count|
+-----------------+-----+
|    United States|    1|
|    United States|  264|
|    United States|   69|
|            Egypt|   24|
|Equatorial Guinea|    1|
+-----------------+-----+



### H2 — SQL `delta.\`path\``

In [0]:
spark.sql(f'SELECT COUNT(*) AS c FROM delta.`{P_BASIC}`').show()

+---+
|  c|
+---+
|510|
+---+



### H3 — `table` after temp view

In [0]:
spark.read.format('delta').load(P_BASIC).createOrReplaceTempView('t_basic'); spark.sql('SELECT MIN(count), MAX(count) FROM t_basic').show()

+----------+----------+
|min(count)|max(count)|
+----------+----------+
|         1|    348113|
+----------+----------+



### H4 — JSON copy for merge lab later

In [0]:
spark.read.format('json').load(SOURCE_JSON).write.format('delta').mode('overwrite').option('overwriteSchema','true').save(P_JSON); print('OK', P_JSON)

OK abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_json


## Part I — `replaceWhere` (conditional overwrite) — discussion + pattern

**Idea:** Overwrite **only** rows matching a predicate (partition pruning). Reduces blast radius vs full `overwrite`.

```python
# Example pattern (uncomment & adapt partition column):
# df.write.format("delta").mode("overwrite").option("replaceWhere", "dest_initial = 'Z'").save(P_PART)
```

Use when you partition by a stable key (date, region). **Wrong predicates** can corrupt data — test in dev first.

### I2 — Hands-on: **`replaceWhere`** on a **single** partition (`P_PART`)

Run **after** Part E. This **overwrites only** rows where `dest_initial = 'U'`. We add a marker column with **`mergeSchema`** so you can see the effect in `DESCRIBE HISTORY`.

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

slice_df = (
    spark.read.format("delta")
    .load(P_PART)
    .where("dest_initial = 'U'")
    .withColumn("partition_refresh_at", current_timestamp())
    .withColumn("replacewhere_note", lit("slice_U_refreshed"))
)
(
    slice_df.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "dest_initial = 'U'")
    .option("mergeSchema", "true")
    .save(P_PART)
)
print("Rows in U slice after replace:", spark.read.format("delta").load(P_PART).where("dest_initial = 'U'").count())
spark.sql(f"DESCRIBE HISTORY delta.`{P_PART}`").select("version", "operation").show(5, truncate=False)

Rows in U slice after replace: 129
+-------+---------+
|version|operation|
+-------+---------+
|1      |WRITE    |
|0      |WRITE    |
+-------+---------+



## Part K — Analytics on **`P_BASIC`** (aggregations & distribution)

In [0]:
from pyspark.sql.functions import col

b = spark.read.format("delta").load(P_BASIC)
b.groupBy("DEST_COUNTRY_NAME").agg({"count": "sum"}).orderBy(col("sum(count)").desc()).show(12, truncate=False)
b.selectExpr("percentile_approx(count, 0.5) as median_cnt", "max(count) as max_cnt", "min(count) as min_cnt").show()

+------------------+----------+
|DEST_COUNTRY_NAME |sum(count)|
+------------------+----------+
|United States     |769864    |
|Canada            |16542     |
|Mexico            |12400     |
|United Kingdom    |3258      |
|Germany           |2784      |
|Japan             |2766      |
|Dominican Republic|2218      |
|Brazil            |1990      |
|The Bahamas       |1806      |
|Colombia          |1570      |
|France            |1548      |
|Jamaica           |1466      |
+------------------+----------+
only showing top 12 rows
+----------+-------+-------+
|median_cnt|max_cnt|min_cnt|
+----------+-------+-------+
|        53| 348113|      1|
+----------+-------+-------+



In [0]:
from pyspark.sql.functions import col

# Top origin countries by total flights
spark.read.format("delta").load(P_BASIC).groupBy("ORIGIN_COUNTRY_NAME").sum("count").orderBy(col("sum(count)").desc()).show(12, truncate=False)

+-------------------+----------+
|ORIGIN_COUNTRY_NAME|sum(count)|
+-------------------+----------+
|United States      |770900    |
|Canada             |16610     |
|Mexico             |12440     |
|United Kingdom     |3006      |
|Germany            |2812      |
|Japan              |2614      |
|Dominican Republic |2300      |
|The Bahamas        |1918      |
|Colombia           |1664      |
|France             |1552      |
|Jamaica            |1514      |
|South Korea        |1242      |
+-------------------+----------+
only showing top 12 rows


## Part L — **`P_JSON`** lifecycle + **schema** vs CSV table

In [0]:
# Ensure P_JSON exists (idempotent)
spark.read.format("json").option("inferSchema", True).load(SOURCE_JSON).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_JSON)
spark.read.format("delta").load(P_JSON).printSchema()
spark.read.format("delta").load(P_JSON).show(5, truncate=False)

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|United States    |Romania            |15   |
|United States    |Croatia            |1    |
|United States    |Ireland            |344  |
|Egypt            |United States      |15   |
|United States    |India              |62   |
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
# Column overlap: JSON vs basic Delta (names only)
cols_j = set(spark.read.format("delta").load(P_JSON).columns)
cols_b = set(spark.read.format("delta").load(P_BASIC).columns)
print("Only in JSON delta:", sorted(cols_j - cols_b))
print("Only in basic delta:", sorted(cols_b - cols_j))
print("Shared:", sorted(cols_j & cols_b))

Only in JSON delta: []
Only in basic delta: ['batch_id', 'ingested_at', 'load_comment', 'source_path']
Shared: ['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count']


## Part M — **Incremental** mindset: watermark column (`ingested_at`)

Production loads often track **last successful timestamp** or **batch id**. Here, filter **newer** rows (demo pattern only):

```python
# last_ts = spark.read.format("delta").load(P_BASIC).selectExpr("max(ingested_at)").collect()[0][0]
# incremental = spark.read.format("csv").options(header=True, inferSchema=True).load(SOURCE_CSV).where(col("ingested_at") > lit(last_ts))  # CSV has no ts — real pipelines add file date
```

Your **`ingested_at`** on `P_BASIC` is **load time**, not event time — use it to teach **metadata lineage**, not business cutoffs.

In [0]:
from pyspark.sql.functions import col

mx = spark.read.format("delta").load(P_BASIC).selectExpr("max(ingested_at) as mx").collect()[0]["mx"]
print("Latest ingested_at in P_BASIC:", mx)
spark.read.format("delta").load(P_BASIC).where(col("batch_id").isNotNull()).select("batch_id", "ingested_at", "DEST_COUNTRY_NAME").show(8, truncate=False)

Latest ingested_at in P_BASIC: 2026-03-21 13:53:34.479984
+---------+--------------------------+-----------------+
|batch_id |ingested_at               |DEST_COUNTRY_NAME|
+---------+--------------------------+-----------------+
|batch_002|2026-03-21 13:53:34.479984|United States    |
|batch_002|2026-03-21 13:53:34.479984|United States    |
|batch_002|2026-03-21 13:53:34.479984|United States    |
|batch_002|2026-03-21 13:53:34.479984|Egypt            |
|batch_002|2026-03-21 13:53:34.479984|Equatorial Guinea|
|batch_002|2026-03-21 13:53:34.479984|United States    |
|batch_002|2026-03-21 13:53:34.479984|United States    |
|batch_002|2026-03-21 13:53:34.479984|Costa Rica       |
+---------+--------------------------+-----------------+
only showing top 8 rows


## Part N — **Quality** checks: nulls & duplicate route keys

In [0]:
from pyspark.sql.functions import col, count

b = spark.read.format("delta").load(P_BASIC)
b.select([count(col(c)).alias(c) for c in ["DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count"]]).show()
b.groupBy("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME").count().where("count > 1").orderBy(col("count").desc()).show(10, truncate=False)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|              510|                510|  510|
+-----------------+-------------------+-----+

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|Marshall Islands |United States      |2    |
|Grenada          |United States      |2    |
|Ireland          |United States      |2    |
|India            |United States      |2    |
|United States    |Costa Rica         |2    |
|United States    |Egypt              |2    |
|United States    |Malta              |2    |
|Sint Maarten     |United States      |2    |
|United States    |Senegal            |2    |
|Singapore        |United States      |2    |
+-----------------+-------------------+-----+
only showing top 10 rows


## Part O — **Table properties** & **detail** for multiple tables

In [0]:
for label, path in [("basic", P_BASIC), ("partitioned", P_PART), ("json", P_JSON)]:
    print("===", label, "===")
    spark.sql(f"DESCRIBE DETAIL delta.`{path}`").select("format", "partitionColumns", "numFiles").show(truncate=False)

=== basic ===
+------+----------------+--------+
|format|partitionColumns|numFiles|
+------+----------------+--------+
|delta |[]              |2       |
+------+----------------+--------+

=== partitioned ===
+------+----------------+--------+
|format|partitionColumns|numFiles|
+------+----------------+--------+
|delta |[dest_initial]  |22      |
+------+----------------+--------+

=== json ===
+------+----------------+--------+
|format|partitionColumns|numFiles|
+------+----------------+--------+
|delta |[]              |1       |
+------+----------------+--------+



In [0]:
spark.sql(f"SHOW TBLPROPERTIES delta.`{P_BASIC}`").show(30, truncate=False)

+-----------------------------+---------+
|key                          |value    |
+-----------------------------+---------+
|delta.enableDeletionVectors  |true     |
|delta.feature.appendOnly     |supported|
|delta.feature.deletionVectors|supported|
|delta.feature.invariants     |supported|
|delta.minReaderVersion       |3        |
|delta.minWriterVersion       |7        |
+-----------------------------+---------+



## Part P — **Worked skill builders** (run each cell; tweak filters)

### P1 — Countries with `count` above global average

In [0]:
from pyspark.sql.functions import avg, col

b = spark.read.format("delta").load(P_BASIC)
avg_c = b.select(avg("count").alias("a")).collect()[0]["a"]
b.where(col("count") > avg_c).select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count").orderBy(col("count").desc()).show(15, truncate=False)
print("avg count:", avg_c)

+-----------------+-------------------+------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count |
+-----------------+-------------------+------+
|United States    |United States      |348113|
|United States    |United States      |348113|
|United States    |Canada             |8305  |
|United States    |Canada             |8305  |
|Canada           |United States      |8271  |
|Canada           |United States      |8271  |
|United States    |Mexico             |6220  |
|United States    |Mexico             |6220  |
|Mexico           |United States      |6200  |
|Mexico           |United States      |6200  |
+-----------------+-------------------+------+

avg count: 1655.956862745098


### P2 — Create temp view + SQL window (rank by destination)

In [0]:
spark.read.format("delta").load(P_BASIC).createOrReplaceTempView("flight_basic")
spark.sql(
    '''
  SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count,
         ROW_NUMBER() OVER (PARTITION BY DEST_COUNTRY_NAME ORDER BY count DESC) AS rnk
  FROM flight_basic
'''
).where("rnk <= 3").show(20, truncate=False)

+-------------------+-------------------+-----+---+
|DEST_COUNTRY_NAME  |ORIGIN_COUNTRY_NAME|count|rnk|
+-------------------+-------------------+-----+---+
|Afghanistan        |United States      |11   |1  |
|Afghanistan        |United States      |11   |2  |
|Angola             |United States      |14   |1  |
|Angola             |United States      |14   |2  |
|Anguilla           |United States      |21   |1  |
|Anguilla           |United States      |21   |2  |
|Antigua and Barbuda|United States      |123  |1  |
|Antigua and Barbuda|United States      |123  |2  |
|Argentina          |United States      |184  |1  |
|Argentina          |United States      |184  |2  |
|Aruba              |United States      |359  |1  |
|Aruba              |United States      |359  |2  |
|Australia          |United States      |290  |1  |
|Australia          |United States      |290  |2  |
|Austria            |United States      |36   |1  |
|Austria            |United States      |36   |2  |
|Azerbaijan 

### P3 — Read **one** partition + cache pattern

In [0]:
from pyspark.sql.functions import col

ud = spark.read.format("delta").load(P_PART).where("dest_initial = 'U'")
print("partition rows:", ud.count())
ud.groupBy("DEST_COUNTRY_NAME").sum("count").orderBy(col("sum(count)").desc()).show(10, truncate=False)

partition rows: 129
+--------------------+----------+
|DEST_COUNTRY_NAME   |sum(count)|
+--------------------+----------+
|United States       |411352    |
|United Kingdom      |2025      |
|United Arab Emirates|320       |
|Uruguay             |43        |
|Ukraine             |14        |
+--------------------+----------+



### P4 — Union CSV-shaped slice with JSON delta (align columns)

In [0]:
from pyspark.sql.functions import lit

a = spark.read.format("delta").load(P_BASIC).select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count").limit(50).withColumn("src", lit("basic"))
j = spark.read.format("delta").load(P_JSON).select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count").limit(50).withColumn("src", lit("json"))
a.unionByName(j).groupBy("src").count().show()

+-----+-----+
|  src|count|
+-----+-----+
|basic|   50|
| json|   50|
+-----+-----+



### P5 — `coalesce` for display (nullable batch metadata)

In [0]:
from pyspark.sql.functions import coalesce, col, lit

spark.read.format("delta").load(P_BASIC).select(
    "DEST_COUNTRY_NAME", coalesce(col("batch_id"), lit("initial_load")).alias("batch_label")
).show(12, truncate=False)

+-----------------+------------+
|DEST_COUNTRY_NAME|batch_label |
+-----------------+------------+
|United States    |initial_load|
|United States    |initial_load|
|United States    |initial_load|
|Egypt            |initial_load|
|Equatorial Guinea|initial_load|
|United States    |initial_load|
|United States    |initial_load|
|Costa Rica       |initial_load|
|Senegal          |initial_load|
|United States    |initial_load|
|Guyana           |initial_load|
|United States    |initial_load|
+-----------------+------------+
only showing top 12 rows


### P6 — `overwrite` the **strict** table with a **smaller** schema file (expect clean load)

In [0]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

small = StructType(
    [
        StructField("DEST_COUNTRY_NAME", StringType(), True),
        StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
        StructField("count", LongType(), True),
    ]
)
spark.read.format("csv").option("header", True).schema(small).load(SOURCE_CSV).limit(100).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_STRICT)
print("rows", spark.read.format("delta").load(P_STRICT).count())

rows 100


### P7 — Compare **row counts** `P_BASIC` vs re-read from **same path** (consistency)

In [0]:
c1 = spark.read.format("delta").load(P_BASIC).count()
c2 = spark.sql(f"SELECT COUNT(*) AS c FROM delta.`{P_BASIC}`").collect()[0]["c"]
print(c1, c2, "match" if c1 == c2 else "MISMATCH")

510 510 match


### P8 — `input_file_name()` **avoid** in UC; use `_metadata` (already in pipeline)

In [0]:
# UC-safe: _metadata.file_path was stored in source_path on ingest
spark.read.format("delta").load(P_BASIC).select("source_path").distinct().show(5, truncate=False)

+-----------------------------------------------------------------------------+
|source_path                                                                  |
+-----------------------------------------------------------------------------+
|abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/2010-summary.csv|
+-----------------------------------------------------------------------------+



### P9 — `mergeSchema` vs `overwriteSchema` (scratch table — does not touch P_BASIC)

In [0]:
# mergeSchema: widen schema on APPEND (additive)
# overwriteSchema: replace schema on OVERWRITE (breaking — use in dev with backups)
from pyspark.sql.functions import lit

P9 = DAY5 + "/delta/p9_schema_demo"
spark.read.format("delta").load(P_BASIC).limit(50).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P9)
spark.read.format("delta").load(P9).limit(5).withColumn("demo_tag", lit("p9")).write.format("delta").mode("append").option("mergeSchema", "true").save(P9)
print("Scratch table schema after widen-append:")
spark.read.format("delta").load(P9).printSchema()

Scratch table schema after widen-append:
root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_path: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- load_comment: string (nullable = true)
 |-- demo_tag: string (nullable = true)



### P10 — `EXPLAIN` on Delta read (predicate pushdown)

In [0]:
spark.sql(f"EXPLAIN EXTENDED SELECT * FROM delta.`{P_BASIC}` WHERE DEST_COUNTRY_NAME = 'Germany'").display(truncate=False)

plan
"== Parsed Logical Plan == 'Project [*] +- 'Filter ('DEST_COUNTRY_NAME = Germany) +- 'UnresolvedRelation [delta, abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_summary_basic], [], false == Analyzed Logical Plan == DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: int, ingested_at: timestamp, source_path: string, batch_id: string, load_comment: string Project [DEST_COUNTRY_NAME#12688, ORIGIN_COUNTRY_NAME#12689, count#12690, ingested_at#12691, source_path#12692, batch_id#12693, load_comment#12694] +- Filter (DEST_COUNTRY_NAME#12688 = Germany) +- SubqueryAlias databricks_training_premium.delta.`abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_summary_basic` +- Relation [DEST_COUNTRY_NAME#12688,ORIGIN_COUNTRY_NAME#12689,count#12690,ingested_at#12691,source_path#12692,batch_id#12693,load_comment#12694] parquet == Optimized Logical Plan == Filter (isnotnull(DEST_COUNTRY_NAME#12688) AND (DEST_COUNTRY_NAME#12688 = Germany)) +- Relation [DEST_COUNTRY_NAME#12688,ORIGIN_COUNTRY_NAME#12689,count#12690,ingested_at#12691,source_path#12692,batch_id#12693,load_comment#12694] parquet == Physical Plan == *(1) ColumnarToRow +- PhotonResultStage +- PhotonScan parquet [DEST_COUNTRY_NAME#12688,ORIGIN_COUNTRY_NAME#12689,count#12690,ingested_at#12691,source_path#12692,batch_id#12693,load_comment#12694] DataFilters: [isnotnull(DEST_COUNTRY_NAME#12688), (DEST_COUNTRY_NAME#12688 = Germany)], DictionaryFilters: [(DEST_COUNTRY_NAME#12688 = Germany)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int,ingested_at:timestamp,source..., RequiredDataFilters: [isnotnull(DEST_COUNTRY_NAME#12688), (DEST_COUNTRY_NAME#12688 = Germany)] == Photon Explanation == The query is fully supported by Photon."


### P11 — Partition pruning: count rows without scanning whole table (partitioned path)

In [0]:
# Should prune to one partition when dest_initial is partition column
spark.read.format("delta").load(P_PART).where("dest_initial = 'A'").groupBy().count().show()
spark.sql(f"EXPLAIN SELECT COUNT(*) FROM delta.`{P_PART}` WHERE dest_initial = 'A'").show(truncate=False)

+-----+
|count|
+-----+
|    9|
+-----+

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                   |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|== Physical Plan ==\nLocalTableScan [count(1)#12819L]\n\n\n== Photon Explanation ==\nPhoton does not fully support the query because:\n\t\tUnsupported node: LocalTableScan [count(1)#12819L].\n\nReference node

### P12 — Compare **`P_JSON`** vs **`P_BASIC`** row counts + sample join on route key

In [0]:
from pyspark.sql.functions import col

jb = spark.read.format("delta").load(P_JSON).alias("j")
bb = spark.read.format("delta").load(P_BASIC).alias("b")
joined = jb.join(bb, on=["DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME"], how="inner")
print("JSON rows:", spark.read.format("delta").load(P_JSON).count())
print("Basic rows:", spark.read.format("delta").load(P_BASIC).count())
print("Inner join rows (sample overlap):", joined.count())
joined.select("j.DEST_COUNTRY_NAME", "j.count", "b.count").limit(8).show(truncate=False)

JSON rows: 256
Basic rows: 510
Inner join rows (sample overlap): 468
+-----------------+-----+-----+
|DEST_COUNTRY_NAME|count|count|
+-----------------+-----+-----+
|United States    |15   |1    |
|United States    |344  |264  |
|United States    |62   |69   |
|Egypt            |15   |24   |
|United States    |1    |25   |
|United States    |62   |54   |
|Costa Rica       |588  |477  |
|Senegal          |40   |29   |
+-----------------+-----+-----+



## Part J — Mini challenges

Complete in the cells below. Compare with others in the class or review answers when the group reconvenes.

### J1 — Write **`DAY5 + '/delta/challenge_distinct'`** as Delta: distinct **`DEST_COUNTRY_NAME`** from `P_BASIC` with a column **`first_seen`** = current_timestamp().

In [0]:
# Your solution:


### J2 — From **`P_PART`**, compute **top 5** `dest_initial` by total **`sum(count)`**.

In [0]:
# Your solution:


### J3 — Using **`DESCRIBE HISTORY`**, identify the **latest** operation on **`P_BASIC`** and read **`operationMetrics`**.

In [0]:
# Your solution:


### J4 — Write SQL only: **`SELECT`** from **`delta.\`P_STRICT\``** (use f-string) — count rows where **`count` > 500**.

In [0]:
# Your solution:


### J5 — Re-read **`P_BASIC`** with **`option('versionAsOf', 0)`** (if version 0 exists). Compare row count to latest — when do they differ?

In [0]:
# Your solution:


### J6 — One paragraph: **Why is `replaceWhere` safer than full `overwrite` on a partitioned table?** When is it still dangerous?

In [0]:
# Your solution:


## Part Q — **Failure** story: bad append + recovery (`mergeSchema` / `overwrite`)

In [0]:
from pyspark.sql import Row

# Try to append a row with wrong type for count — expect failure without casting
bad_type = spark.createDataFrame([Row(DEST_COUNTRY_NAME="ZZ", ORIGIN_COUNTRY_NAME="YY", count="not_a_number")])
try:
    bad_type.write.format("delta").mode("append").save(P_STRICT)
except Exception as e:
    print("Expected type / parse issue:", type(e).__name__)
    print(str(e)[:200])

Expected type / parse issue: AnalysisException
[DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'count' and 'count'.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.schema.


## Next notebooks (Day 5)

1. **`03-Day5-Delta-History-Optimize-Advanced.ipynb`** — time travel, restore concepts, **OPTIMIZE**, **ZORDER**, **VACUUM**, CDF intro.
2. **`04-Day5-Delta-DML-MERGE-SCD.ipynb`** — **MERGE**, **UPDATE**, **DELETE**, **SCD Type 1 / 2** patterns.

---

### Cleanup (optional)

```python
# from pyspark.sql import SparkSession
# dbutils.fs.rm(DAY5.replace("abfss://", "/dbfs/..."), True)  # only if you use DBFS — prefer Azure portal / lifecycle policy for abfss
```